# Collaborative Filtering Recommendation Engine with Embedding Latent Factors

## Overview
This notebook builds an end-to-end collaborative filtering recommendation system using FastAI and the MovieLens Sample dataset. It trains an embedding-based neural network to predict continuous user ratings for movies via matrix factorization representations.

## Key Technical Components
* **Data Ingestion & Matrix Setup:** Uses `CollabDataLoaders.from_csv` to ingest user-item interaction pairs (`ratings.csv`), structuring sparse user and item identifiers into tabular interaction batches.
* **Embedding Model Architecture:** Instantiates a collaborative filtering learner via `collab_learner`, mapping discrete user and movie IDs into continuous latent factor embedding spaces.
* **Constrained Output Activation (`y_range`):** Employs a scaled sigmoid output range `y_range=(0.5, 5.5)` to mathematically constrain model predictions strictly within the valid target rating domain (0.5 to 5.0 stars) while allowing gradient flow at extreme bounds.
* **Optimization & Training:** Applies transfer learning and discriminative learning rate schedules via `learn.fine_tune(10)` to optimize latent user and item factor matrices.
* **Prediction & Inspection:** Utilizes `learn.show_results()` to compare predicted rating scalars directly against ground-truth targets.

------
------
------

# Collaborative Filtering - recommendation system

Collaborative Filtering is a recommendation technique that predicts a user's preferences by analyzing patterns in the behaviour of similar users or items.

## Core Concept: "Wisdom of the Crowd"
In CF, we dont necessarily need to know if a movie is an "Action" movie or if a user lives in "New York". We only care about "who liked what".

## There are 2 main ways to look at it:
1. User-based: "Users who are similar to you also bought..."
2. Item-based: "Users who bought this item also bought..."

## How it Works(The "Latent Factors")
In DL, we handle these using 'Embeddings'. We represent every user and every item as a vector of numbers(latent factors).
* User Embedding: Represents the user's "taste" (e.g. how much they like sci-fi, comedy etc.)
* Item Embedding: Represents the item's(movie's) "features" (e.g. how much sci-fi, comedy etc. vibes it has)
When we multiply these two vectors together(a dot product), we get a predicted rating.
            Predicted rating $$\text{Predicted Rating} \approx q_i^T p_u$$
  Where:
  * $q_i$ is the vector for item i.
  * $p_u$ is the vector for user u.


## Why this is different from Tabular DL?
Even though the data often looks like a table(User ID, Movie ID, Rating), it's unique because:
* Sparsity: Most users haven't seen most movies. Our matrix is like 99% empty.
* Cold Start Problem: New users or new items have no history, making it hard to recommended anything initially.

## The Deep Learning Twist
While traditional CF used **Matrix Factorization**, modern DL approaches use **Neural Collaborating Filtering(NCF)**. Instead of just a simple dot product, we feed those embeddings into a NN(MLP-multi layer perceptron) to find complex, non-linear patterns in how users interact with items.

In [1]:
!pip install -Uqq fastai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.1/235.1 kB 5.3 MB/s eta 0:00:00a 0:00:01


In [5]:
from fastai.collab import *
from fastai.data.all import * # this line is for untar_data and URLs

# vision.all imports ResNet, ImageDataLoaders, aug_transforms while 
# tabular.all imports tabular_learner, procs like Categorify and FillMissing, optimization logic like fit_one_cycle
# It imports CollabDataLoaders, collab_learner, EmbeddingDotBias

# The Data Prep
path = untar_data(URLs.ML_SAMPLE)
dls = CollabDataLoaders.from_csv(path/'ratings.csv')

# URLs.ML_SAMPLE is the small subset of the famous MovieLens dataset. It contains three main columns: userId, movieId, and rating.
# CollabDataLoaders: Unlike TabularDataLoaders where we had to list out our columns, this factory method is "opinionated". It looks
#   at the .csv and assumes Column 0: userId, Column 1: movieId(movies), Column 2: rating(targets)

# It automatically creates a "Vocab" for users and movies. It maps our IDs(like User #423) to continuous integers (like 0,1,2...) so they
#   can be looked up in an Embedding Matrix.

In [6]:
dls.show_batch()
# It shows a dataframe snippet of what the model is "eating" during training. Generally used for verification of data.

,userId,movieId,rating
0,654,1270,4.5
1,665,2028,4.0
2,380,4886,4.0
3,615,1265,3.5
4,199,1136,5.0
5,564,736,5.0
6,607,595,2.0
7,387,1923,5.0
8,402,5952,4.0
9,388,2762,4.0


In [9]:
# The model construction
learn = collab_learner(dls, y_range=(0.5,5.5))
'''
* y_range(The Sigmoid Trick): This is crucial. NN are naturally good at producing any numbers between -∞ to +∞. But our
  rating is strictly between 1 and 5(5 point recommended system).
  - By setting y_range=(0.5,5.5) fastai forces the final output through a Sigmoid function.
  - Why 0.5 to 5.5? Sigmoid curve flattens out at the ends. If we use 1 to 5 we'll never almost get 1 and 5 exactly so pushing the range 
    slightly wider.
* The Architecture: use_nn
  - "use_nn = False" (Default) : It uses a Dot Product model. Its fast, interpretable for similar movies.
  - "use_nn = True" : It switches to NN architecture(MLP). It takes the embeddings, concatenates them, and passes them through dense
     layers. This is better for capturing complex, non-linear relationships.

     Here as we haven't specified(use_nn = True), EmbeddingDotBias model is built. 
     Table is created "User Weights", "Movie Weights" then Dot Product.
     Also adds:
     Bias: It adds a "User Bias" (some people rate everything low) and a "Movie Bias" (some movies are just universally loved)
'''


# The Training
learn.fine_tune(10)

# Here is no pre-trained backbone, fine_tune still performs its magic: it finds a good learning rate, trains the head, and then trains
# the whole thing. In CF, this is just a robust way to ensure the embeddings converge quickly.

# Here vaild_loss is the mean square error, root(0.71) = 0.8426

epoch,train_loss,valid_loss,time
0,1.502156,1.363125,00:00


epoch,train_loss,valid_loss,time
0,1.389595,1.314431,00:00
1,1.295388,1.169211,00:00
2,1.065785,0.904364,00:00
3,0.830915,0.764711,00:00
4,0.674731,0.727720,00:00
5,0.642540,0.716399,00:00
6,0.603644,0.713625,00:00
7,0.632131,0.711695,00:00
8,0.596724,0.710907,00:00
9,0.602719,0.710809,00:00


In [10]:
# By default collab_learner sets n_factors=50. This means every movie is represented by 50 numbers.
# like:
#     Factor1: "How much Explosions are these?"
#     Factor2: "How Romantic is this?"
#     Factor50: "Is it have a 90s feel?"

learn.show_results() 

,userId,movieId,rating,rating_pred
0,39.0,90.0,4.5,3.999898
1,4.0,45.0,4.5,4.318060
2,3.0,43.0,3.0,3.582444
3,35.0,40.0,2.0,2.920418
4,47.0,58.0,5.0,4.197245
5,2.0,72.0,0.5,3.203435
6,47.0,16.0,4.5,4.729111
7,72.0,82.0,2.5,3.392178
8,16.0,45.0,4.0,4.238972
